# Capstone Project: Deep RL Trading Agent

## Introduction

Here is the structure of the project Python source code: 

- `task.py`: the environment definition in this file.
- `agents/`: Folder containing reinforcement learning agents.
    - `buyandholdagent.py`: Benchmark agent.
    - `deeprlagent.py`: the deep rl trading agent.
- `market.py`: This file contains the interface to the stock market data.
- `data/`: stock prices.
- `resources/`: additional files.

## Data Loading and Data Cleansing of finance data
The data set is constructed from the raw data files.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import pandas as pd
from market import Market
from task import Task



# loading the stock prices
# all trade data is coming from XETRA

df_usinterest =  pd.read_csv('data/FRB_H15.csv', header=5, names=['day','US_rate'], skiprows=0, parse_dates=[0],\
                          infer_datetime_format=True)

df_sp500_prices = pd.read_csv('data/Investing_622391_historic_data.csv', header=0,\
                           names=['day','closing', 'opening', 'high', 'low', 'vol', 'change'],\
                           skiprows=0, parse_dates=[0], infer_datetime_format=True, decimal=',')

df_eurostoxx600_prices = pd.read_csv('data/Investing_DBX1A7_historic_data.csv', header=0,\
                           names=['day','closing', 'opening', 'high', 'low', 'vol', 'change'],\
                           skiprows=0, parse_dates=[0], infer_datetime_format=True, decimal=',')

df_msciworld_prices = pd.read_csv('data/Investing_ETF110_historic_data.csv', header=0,\
                           names=['day','closing', 'opening', 'high', 'low', 'vol', 'change'],\
                           skiprows=0, parse_dates=[0], infer_datetime_format=True, decimal='.')

# convert vol column to float dtype
#1. remove K, M, -
#2. replace , with .
#3. parse as float
#4. multiply by 1000, 1000000
def vol2float(x):
    if (x.find('-')!=-1):
        return float(0.0)
    
    if (x.find('K')!=-1):
        return float(x.replace('K','').replace(',','.'))*1000.00
    
    if (x.find('M')!=-1):
        return float(x.replace('M','').replace(',','.'))*1000000.00
    
    return float('nan')

df_sp500_prices['SP500_volno'] = df_sp500_prices['vol'].apply(vol2float)
df_eurostoxx600_prices['ESTOXX_volno'] = df_eurostoxx600_prices['vol'].apply(vol2float)
df_msciworld_prices['MSCI_volno'] = df_msciworld_prices['vol'].apply(vol2float)

# remove records without closing price
df_sp500_prices = df_sp500_prices.dropna(subset=['closing'])
df_eurostoxx600_prices = df_eurostoxx600_prices.dropna(subset=['closing'])
df_msciworld_prices = df_msciworld_prices.dropna(subset=['closing'])

# rename price and volumn columns
df_sp500_prices=df_sp500_prices.rename(columns={"closing": "SP500_price"})
df_eurostoxx600_prices = df_eurostoxx600_prices.rename(columns={"closing": "ESTOXX_price"})
df_msciworld_prices = df_msciworld_prices.rename(columns={"closing": "MSCI_price"})


# create calendar with all days
calendar_series = pd.Series(pd.date_range('2009-10-16', '2019-10-16', freq='D', name='date'))
calendarweekday = calendar_series.dt.dayofweek
calendarweekday.name='dayofweek'
calendarmonth = calendar_series.dt.month
calendarmonth.name='month'
calendarday = calendar_series.dt.day
calendarday.name='dayno'
calendarmonthstart = calendar_series.dt.is_month_start
calendarmonthstart.name='monthstart'

df_calendar=pd.concat([calendar_series,calendarweekday,calendarmonth, calendarday,calendarmonthstart], axis=1)

# join each dataframe with calendar and with interest rate
dfdata=pd.merge(df_calendar,df_sp500_prices,how='left',left_on='date', right_on='day')
dfdata=pd.merge(dfdata,df_eurostoxx600_prices,how='left',left_on='date', right_on='day')
dfdata=pd.merge(dfdata,df_msciworld_prices,how='left',left_on='date', right_on='day')
dfdata=pd.merge(dfdata,df_usinterest,how='left',left_on='date', right_on='day')

# remove surplus columns
dfdata = dfdata[['date','dayofweek','month','dayno','monthstart','SP500_price', 'SP500_volno', \
                 'ESTOXX_price', 'ESTOXX_volno', 'MSCI_price', 'MSCI_volno', 'US_rate']]

# remove records without closing price
dfdata = dfdata.dropna(subset=['SP500_price', 'SP500_volno', 'ESTOXX_price', 'ESTOXX_volno', 'MSCI_price', 'MSCI_volno' ])

dfdata.dtypes

## Data Exploration and Visualization of finance data
### Basic statistic information

Please see the column names and basic statistic information on the used finance data below.

In [ ]:
from IPython.display import display # Allows the use of display() for DataFrames

display(dfdata.describe())
print(dfdata.head(10))

### Abnormalities

For detecting outliers in the data, the Tukey's Method will be used: An outlier step is calculated as 1.5 times the interquartile range (IQR). A data point with a feature that is beyond an outlier step outside of the IQR for that feature is considered abnormal.

In [ ]:
import numpy as np

financialdfdata = dfdata[['SP500_price', 'SP500_volno','ESTOXX_price', 'ESTOXX_volno', 'MSCI_price', 'MSCI_volno', 'US_rate']]

l_outliers=[]
# For each feature find the data points with extreme high or low values
for feature in financialdfdata.keys():
    
    # TODO: Calculate Q1 (25th percentile of the data) for the given feature
    Q1 = np.percentile(financialdfdata[feature],25)
    
    # TODO: Calculate Q3 (75th percentile of the data) for the given feature
    Q3 = np.percentile(financialdfdata[feature],75)
    
    # TODO: Use the interquartile range to calculate an outlier step (1.5 times the interquartile range)
    step = (Q3-Q1)*1.5    
    
    # Display the outliers
    print("Data points considered outliers for the feature '{}':".format(feature))
    display(financialdfdata[~((financialdfdata[feature] >= Q1 - step) & (financialdfdata[feature] <= Q3 + step))])
    for elem in financialdfdata[~((financialdfdata[feature] >= Q1 - step) & (financialdfdata[feature] <= Q3 + step))].index:
        l_outliers.append(elem)

#  print list and remove the duplicate indices
l_uniqueoutliers= sorted(list(dict.fromkeys(l_outliers)))
print ("Sorted list of unique outlier indices {}".format(l_uniqueoutliers))



### Exploratory visualization
Please see below the scatter plot, the correlation matrix and the time series of the finance data and US interest rate.

In [ ]:
%matplotlib inline
pd.plotting.scatter_matrix(financialdfdata, alpha = 0.3, figsize = (14,8), diagonal = 'kde')
financialdfdata.corr()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

plt.plot(dfdata['date'], dfdata['SP500_price'] , color='green', label='S&P 500')
plt.plot(dfdata['date'], dfdata['ESTOXX_price'] , color='blue', label='Euro STOXX 600')
plt.plot(dfdata['date'], dfdata['MSCI_price'] , color='red', label='MSCI World')

#plt.plot(eps, rews2, color='black')

plt.xlabel('Date')
plt.ylabel('stock price (?)')
plt.legend()
plt.draw()
plt.show()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

plt.plot(dfdata['date'], dfdata['US_rate'] , color='green')

plt.xlabel('Date')
plt.ylabel('US interest rate (%)')
plt.draw()
plt.show()

## Setting up the environment
The following cell splits the data in a training and test set and instantiates the market and task objects. The market object creates an interface to the stock prices and interest rates (i.e. market data).

In [ ]:
%load_ext autoreload
%autoreload 2

# create test /validation data
dfdata_norecords = int(round(len(dfdata.index)*0.75))
dfdata_train = dfdata.iloc[0:dfdata_norecords,:]
dfdata_test = dfdata.iloc[dfdata_norecords:,:]

#create markets
market_train = Market(dfdata_train)
market_test = Market(dfdata_test)

# create tasks
monthly_allocation =  1000.0  #monthly upper spending limit
starting_budget =  120000.0  

task_train = Task(market_train, monthly_allocation, starting_budget, ['SP500'], 'SP500',-10.0, -5000.0)
task_test = Task(market_test, monthly_allocation, starting_budget, ['SP500'], 'SP500',-10.0, -5000.0)

labels = ['episode', 'time', 'price', 'action', 'portfolio', 'reward', 'total value']


## The Task

The `__init__()` method is used to initialize several variables that are needed to specify the task.  
- The market data comes from the `Market` class (from `market.py`).  
- Action repeats are not sensible for this use case, so their code was removed from the original implementation. 
- We set the number of elements in the state vector (`state_size`).  For the sample task, we only work with the 6-dimensional pose information.  
- The environment will always have a 1-dimensional action space, with the floating point entry encoding the transaction volume (`action_size=1`). We also set the minimum (`action_low`) and maximum (`action_high`) values of each entry here.
- The task in this provided file is for the agent to maximize the profit during the trading period.  We specify that target position as a variable.

The `reset()` method resets the market status.

The `step()` method is perhaps the most important.  It accepts the agent's choice of action `transactions`, which is used to prepare the next state to pass on to the agent.  Then, the reward is computed from `get_reward()`.  The episode is considered done if the simulation date is outside the time interval of available market data.

## Running the benchmark agent

The agent given in `agents/buyandholdagent.py` uses a fixed action that is executed at the beginning of every month. The agent does not learn and is only there as a benchmark for the more sophisticated `agents/deeprlagent.py`.

The following code runs the agent and produces a baseline for comparison.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
from agents.buyandholdagent import BAH_Agent
from task import Task
import numpy as np
import csv
from datetime import datetime
import copy
from market import Market

dateTimeObj = datetime.now()
timestampStr = dateTimeObj.strftime("%d-%b-%Y_%H%M%S")

file_output = 'data_bah_'+timestampStr+'.csv'          # file name for saved results
done = False
rewards_bench = []  # to store the total reward per episode
monthly_action = {'SP500': monthly_allocation - 1.0} #monthly investment of 1000? in SP500
maxresults_bench = {x : [] for x in labels}  # to store the transactions of the best episode
maxreward = -10000

agent = BAH_Agent(task_train, task_test, monthly_action) 

def report_evaluation(task:Task, reward:float):
    print('Time: {0:5d}'.format(task.market.currentdate))
    print('Portfolio: ' + str(task.portfolio))
    print('Total Value: {0:.2f}?'.format(task.get_total_value()))
    print('Total reward: {0:.2f}'.format(reward))

# training runs
with open(file_output, 'w', newline='\n', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(labels)
    state = agent.reset_episode() # start a new episode, also resets task     
    results = {x : [] for x in labels}  # to store the trajectory of the best episode

    while True:
        action = agent.act(state) # delivers an action for the current state
        next_state, reward, done = task_train.step(action) # push the simulation further
          
        agent.step(action, reward, next_state, done)
            
        # store the details of the current episode
        to_write = [1]+[task_train.market.currentdate] + [task_train.market.get_value('SP500_price')] + [action[0]] \
        + [task_train.portfolio['SP500']] +[reward] + [task_train.get_total_value()]            
        
        writer.writerow(to_write)
        for ii in range(len(labels)):
            results[labels[ii]].append(to_write[ii])            
                        
        state = next_state               

        if done:
            rewards_bench.append((1, reward, task_train.market.currentdate, task_train.get_total_value())) #keep track of the total reward            
            if reward>=maxreward: #save the results of the best run so far
                maxreward=max(reward,maxreward)
                maxresults_bench = copy.deepcopy(results) #save the best run                
            break
sys.stdout.flush()

print('Best episode in the training results: ')
print('Time: {0:5d}'.format(maxresults_bench['time'][-1]))
print('Portfolio: {0:.2f}?'.format(maxresults_bench['portfolio'][-1]))
print('Total Value: {0:.2f}?'.format(maxresults_bench['total value'][-1]))
print('Total reward: {0:.2f}'.format(maxresults_bench['reward'][-1]))
print('\n')

# test runs
state = agent.reset_episode(True) # start a new episode in the test mode  
while True:
    action = agent.act(state) # delivers an action for the current state
    next_state, reward, done = task_test.step(action) # push the simulation further    
    state = next_state               
    if done:
        print('Test results: ')
        report_evaluation(task_test, reward)
        break
sys.stdout.flush()


## Running the deep reinforcement agent

The agent given in `agents/deeprlagent.py` is executed and the results are evaluated.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
from agents.deeprlagent import DeepRL_Agent
from task import Task
import numpy as np
import csv
from datetime import datetime
import copy
from market import Market

dateTimeObj = datetime.now()
timestampStr = dateTimeObj.strftime("%d-%b-%Y_%H%M%S")

file_output = 'data_deeprl_'+timestampStr+'.csv'          # file name for saved results
done = False
rewards_dl = []  # to store the total reward per episode
maxresults_dl = {x : [] for x in labels}  # to store the transactions of the best episode
num_episodes = 100
maxreward = -10000

agent = DeepRL_Agent(task_train, task_test) 

#################################
# training runs
#################################
with open(file_output, 'w', newline='\n', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(labels)
    for i_episode in range(1, num_episodes+1):
        state = agent.reset_episode() # start a new episode, also resets task     
        results = {x : [] for x in labels}  # to store the trajectory of the best episode

        while True:
            action = agent.act(state) # delivers an action for the current state
            next_state, reward, done = task_train.step(action) # push the simulation further
            
            agent.step(action, reward, next_state, done)
            
            # store the details of the current episode
            to_write = [i_episode]+[task_train.market.currentdate] + [task_train.market.get_value('SP500_price')]\
            + [action[0]] + [task_train.portfolio['SP500']] +[reward] + [task_train.get_total_value()]            
            writer.writerow(to_write)
            for ii in range(len(labels)):
                results[labels[ii]].append(to_write[ii])            
                        
            state = next_state               

            if done:
                rewards_dl.append((i_episode, reward, task_train.market.currentdate, task_train.get_total_value())) 
                if reward>=maxreward: #save the results of the best run so far
                    maxreward=max(reward,maxreward)
                    maxresults_dl = copy.deepcopy(results) #save the best run
                break
    sys.stdout.flush()
    
print('Best episode in the training results: ')
print('Time: {0:5d}'.format(maxresults_bench['time'][-1]))
print('Portfolio: {0:.2f}?'.format(maxresults_dl['portfolio'][-1]))
print('Total Value: {0:.2f}?'.format(maxresults_dl['total value'][-1]))
print('Total reward: {0:.2f}'.format(maxresults_dl['reward'][-1]))
print('\n')

##########################
# test run
##########################
def report_evaluation(task:Task, reward:float):
    print('Time: {0:5d}'.format(task.market.currentdate))
    print('Portfolio: ' + str(task.portfolio))
    print('Total Value: {0:.2f}?'.format(task.get_total_value()))
    print('Total reward: {0:.2f}'.format(reward))
    
file_output = 'data_deeprl_test_'+timestampStr+'.csv'          # file name for saved results
done = False
state = agent.reset_episode(True) # start a new episode in the test mode  

# training runs
with open(file_output, 'w', newline='\n', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(labels)    
    while True:
        action = agent.act(state, True) # delivers an action for the current state
        next_state, reward, done = task_test.step(action) # push the simulation further        
        
        # store the details of the current episode
        to_write = [i_episode]+[task_test.market.currentdate] + [task_test.market.get_value('SP500_price')]\
            + [action[0]] + [task_test.portfolio['SP500']] +[reward] + [task_test.get_total_value()]            
        writer.writerow(to_write)
        for ii in range(len(labels)):
            results[labels[ii]].append(to_write[ii])            
                        
        state = next_state               

        if done:
            print('Test results: ')
            report_evaluation(task_test, reward)
            break
        sys.stdout.flush()

## Plot the Rewards
### Learning progress (rewards)
The following graph shows the total rewards per episode as a graph. The red line is the benchmark and the black line shows the learning progress of the deeprl agent.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

eps2, rews2, dur2, totval2 = np.array(rewards_dl).T
rews=np.full_like(rews2,rewards_bench[0][1])

plt.plot(eps2, rews, color='red', label='Benchmark')
plt.plot(eps2, rews2, color='black', label='DeepRL Agent')

plt.xlabel('Episode')
plt.ylabel('Reward')
plt.legend()
plt.draw()
plt.show()

### Benchmark agent: Transactions over the best training period

In [ ]:
dfmaxresults = pd.DataFrame.from_dict(maxresults_bench)

selling_sign = []
buying_sign = []
for indx,rowelem in dfmaxresults.iterrows():
    if rowelem['action']<0.0:
        selling_sign.append(int(rowelem['time'])-1)    
    if rowelem['action']>0.0:    
        buying_sign.append(int(rowelem['time'])-1)

fig = plt.figure(figsize = (15,5))
plt.plot(dfmaxresults['time'], dfmaxresults['price'], color='b', lw=2., label='S&P 500 stock price')
plt.plot(dfmaxresults['time'], dfmaxresults['price'], '^', markersize=10, color='m', label = 'buying signal', markevery = buying_sign)
plt.plot(dfmaxresults['time'], dfmaxresults['price'], 'v', markersize=10, color='k', label = 'selling signal', markevery = selling_sign)
plt.legend()
plt.show()


### DeepRL agent: Transactions over the best training period

In [ ]:
dfmaxresults = pd.DataFrame.from_dict(maxresults_dl)

selling_sign = []
buying_sign = []
for indx,rowelem in dfmaxresults.iterrows():
    if rowelem['action']<0.0:
        selling_sign.append(int(rowelem['time'])-1)    
    if rowelem['action']>0.0:    
        buying_sign.append(int(rowelem['time'])-1)

fig = plt.figure(figsize = (15,5))
plt.plot(dfmaxresults['time'], dfmaxresults['price'], color='b', lw=2., label='S&P 500 stock price')
plt.plot(dfmaxresults['time'], dfmaxresults['price'], '^', markersize=10, color='m', label = 'buying signal', markevery = buying_sign)
plt.plot(dfmaxresults['time'], dfmaxresults['price'], 'v', markersize=10, color='k', label = 'selling signal', markevery = selling_sign)
plt.legend()
plt.show()
